
# HAA + 주도섹터 + 수급빈집 시스템 v2.0

v2.0은 v0.x~v1.x에서 만든 기능을 **유지보수 가능한 하나의 시스템**으로 다시 정리한 버전입니다.

핵심 원칙:

- 설정은 한 곳
- 연구와 실전 운영 분리
- 저장/캐시/로그 공통화
- 중복 함수 최소화
- 실전 진입점은 `run_daily()`
- 연구 진입점은 `run_research()`

---

## 시스템 구조

### Research Mode
HAA  
→ 과거 ETF universe  
→ 과거 주도섹터  
→ ETF Core  
→ ETF 구성종목 스냅샷  
→ 수급 Alpha  
→ Walk-forward OOS  
→ 파라미터 안정성

### Daily Mode
HAA  
→ Fear & Greed  
→ KOSPI 현금비중  
→ 오늘의 주도섹터  
→ ETF Core  
→ 수급 Alpha  
→ 목표 포트폴리오  
→ 승인 주문  
→ NAV  
→ 운영 리포트  
→ 저장


In [ ]:
%pip install -q -U pykrx yfinance finance-datareader

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.2 MB/s eta 0:00:00


In [ ]:
import os
from getpass import getpass

os.environ["KRX_ID"] = input("KRX ID: ")
os.environ["KRX_PW"] = getpass("KRX Password: ")

print("KRX credentials loaded into environment.")

KRX ID: hyunsungkim73
KRX Password: ··········
KRX credentials loaded into environment.


In [ ]:
from pykrx import stock
import pykrx

print("pykrx loaded")

KRX 로그인 시도...
  로그인 ID: hyunsungkim73
KRX 로그인 완료.
  로그인 시간: 2026-08-22 22:29:53
  만료 시간: 2026-08-22 23:29:53
pykrx loaded


In [ ]:
date = "20260821"

try:
    etfs = stock.get_etf_ticker_list(date)

    print("ETF count:", len(etfs))
    print("First 10:", etfs[:10])

except Exception as e:
    print("KRX ETF TEST FAILED")
    print(type(e).__name__, e)

ETF count: 1161
First 10: ['0193M0', '466810', '457930', '0218K0', '487750', '445690', '0120J0', '0112X0', '465780', '0191M0']


# 0. 설치

In [ ]:
!pip install -q -U pip
!pip install -q -U pykrx yfinance finance-datareader

# 1. 공통 Import

In [ ]:

import os
import math
import json
import time
import traceback
import warnings
import itertools
from pathlib import Path
from datetime import datetime, timedelta
from functools import wraps

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import yfinance as yf

from pykrx import stock
import FinanceDataReader as fdr

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# 2. CONFIG — 여기만 주로 수정

In [ ]:

CONFIG = {
    # -------------------------
    # Portfolio
    # -------------------------
    "current_cash": 0,
    "leader_sector_count": 5,
    "etf_core_within_stock": 0.35,
    "alpha_within_stock": 0.65,

    # -------------------------
    # Stock alpha
    # -------------------------
    "alpha_top_n_per_sector": 3,
    "max_constituents_per_etf": 15,
    "max_single_stock_weight": 0.05,
    "min_stock_trading_value": 1_000_000_000,

    # -------------------------
    # ETF
    # -------------------------
    "min_etf_trading_value": 500_000_000,

    # -------------------------
    # Execution
    # -------------------------
    "max_daily_turnover": 0.20,
    "max_new_buys": 5,
    "max_sector_weight": 0.25,
    "min_trade_value": 200_000,
    "cash_tolerance": 0.02,

    # -------------------------
    # Cost
    # -------------------------
    "core_cost_rate": 0.0010,
    "alpha_cost_rate": 0.0020,

    # -------------------------
    # HAA
    # -------------------------
    "haa_canary": "TIP",
    "haa_offensive": ["SPY","IWM","VEA","VWO","VNQ","DBC","IEF","TLT"],
    "haa_defensive": ["BIL","IEF"],

    # -------------------------
    # Fear & Greed
    # -------------------------
    "use_cnn_fear_greed": True,
    "manual_fear_greed": None,
    "fg_greed_level": 75,
    "fg_extreme_level": 85,
    "fg_max_risk_level": 90,

    # -------------------------
    # Alpha weights
    # -------------------------
    "flow_window_weights": {5:0.20, 20:0.50, 60:0.30},
    "investor_weights": {"foreign":0.50, "institution":0.50},
    "alpha_weights": {
        "supply_empty":0.30,
        "price_momentum":0.25,
        "trend":0.15,
        "liquidity":0.10,
        "high_proximity":0.10,
        "pullback":0.10,
    },

    # -------------------------
    # Cache / Storage
    # -------------------------
    "use_google_drive": True,
    "api_max_retries": 3,
    "api_retry_sleep_sec": 1.5,
    "cache_ttl_hours": 12,
    "snapshot_lookback_days": 45,
    "snapshot_fallback_to_current": True,

    # -------------------------
    # Research Walk-forward
    # -------------------------
    "wf_train_months": 18,
    "wf_test_months": 6,
    "wf_min_snapshot_coverage": 0.70,
}


# 3. 섹터 사전

In [ ]:

EXCLUDE_KEYWORDS = [
    "인버스","레버리지","2X","2x","곱버스",
    "채권","국고채","국채","회사채","단기","CD금리",
    "KOFR","머니마켓","MMF","액티브채권",
    "금선물","은선물","원유선물","달러선물",
]

SECTOR_KEYWORDS = {
    "반도체":["반도체","AI반도체","시스템반도체","반도체소부장"],
    "전력":["전력","전력기기","전력인프라","AI전력","전력망","그리드"],
    "원전":["원전","원자력","SMR"],
    "방산":["방산","K방산","우주방산","방위산업"],
    "로봇":["로봇","휴머노이드"],
    "조선":["조선","조선해운"],
    "우주":["우주","우주항공","스페이스"],
    "통신":["통신","네트워크","5G","광통신","통신장비"],
    "바이오":["바이오","헬스케어","제약","의료"],
    "밸류업":["밸류업","고배당","주주환원"],
    "금융":["은행","금융","보험","증권"],
    "자동차":["자동차","모빌리티","전기차"],
    "2차전지":["2차전지","이차전지","배터리"],
    "화장품":["화장품","K뷰티","K-뷰티"],
    "인터넷게임":["게임","인터넷","플랫폼"],
    "엔터":["엔터","미디어","KPOP","K-POP"],
    "건설":["건설","건설기계","인프라"],
    "친환경":["수소","태양광","친환경","클린에너지"],
}

US_SECTOR_ETFS = {
    "반도체":["SMH","SOXX"],
    "전력":["XLU","GRID"],
    "원전":["URA","NLR"],
    "방산":["ITA","XAR"],
    "로봇":["BOTZ","ROBO"],
    "조선":["BOAT"],
    "우주":["ARKX","UFO"],
    "통신":["XLC","IYZ"],
    "바이오":["XBI","IBB"],
    "밸류업":["VYM","SCHD"],
    "금융":["XLF","KBE"],
    "자동차":["CARZ"],
    "2차전지":["LIT"],
    "화장품":["XLY"],
    "인터넷게임":["HERO"],
    "엔터":["XLC"],
    "건설":["XLI"],
    "친환경":["ICLN"],
}


# 4. Storage / Cache

In [ ]:

def setup_storage():
    drive_available = False

    if CONFIG["use_google_drive"]:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            drive_available = True
        except:
            drive_available = False

    base = (
        Path("/content/drive/MyDrive/HAA_Leader_System")
        if drive_available
        else Path("/content/HAA_Leader_System")
    )

    for sub in [
        "portfolio","snapshots","logs","reports",
        "rebalancing","cache","etf_constituents"
    ]:
        (base / sub).mkdir(parents=True, exist_ok=True)

    return base

BASE_DIR = setup_storage()
CACHE_DIR = BASE_DIR / "cache"

for sub in ["krx_price","krx_flow","krx_mcap","yahoo"]:
    (CACHE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)


BASE_DIR: /content/HAA_Leader_System


# 5. 공통 Utility

In [ ]:

def retry_call(func, *args, retries=None, sleep_sec=None, **kwargs):
    retries = retries or CONFIG["api_max_retries"]
    sleep_sec = sleep_sec or CONFIG["api_retry_sleep_sec"]

    last_error = None

    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            last_error = e
            if attempt < retries:
                time.sleep(sleep_sec * attempt)

    raise last_error


def cache_is_fresh(path, ttl_hours=None):
    ttl_hours = ttl_hours or CONFIG["cache_ttl_hours"]
    p = Path(path)

    if not p.exists():
        return False

    return (time.time() - p.stat().st_mtime) <= ttl_hours * 3600


def safe_write_csv(df, path, index=True):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=index, encoding="utf-8-sig")
    tmp.replace(path)


def download_close(tickers, period="1y"):
    tickers = list(tickers)
    if not tickers:
        return pd.DataFrame()

    raw = yf.download(
        tickers,
        period=period,
        auto_adjust=True,
        progress=False,
        threads=True
    )

    if raw.empty:
        return pd.DataFrame()

    if isinstance(raw.columns, pd.MultiIndex):
        close = raw["Close"].copy()
    else:
        close = raw[["Close"]].copy()
        close.columns = tickers[:1]

    if isinstance(close, pd.Series):
        close = close.to_frame()

    return close.dropna(how="all")


def return_n_days(s, n):
    s = s.dropna()
    if len(s) <= n:
        return np.nan
    return s.iloc[-1] / s.iloc[-(n+1)] - 1


def momentum_13612u(s):
    vals = [return_n_days(s, n) for n in [21,63,126,252]]
    vals = [v for v in vals if pd.notna(v)]
    return np.mean(vals) if vals else np.nan


def find_col(df, keyword):
    for c in df.columns:
        if keyword in str(c):
            return c
    return None


def safe_pct_rank(series):
    s = pd.Series(series)
    return s.rank(pct=True) if s.notna().sum() else pd.Series(np.nan, index=s.index)


def classify_sector_from_name(name):
    name_u = str(name).upper()
    for sector, kws in SECTOR_KEYWORDS.items():
        for kw in kws:
            if str(kw).upper() in name_u:
                return sector
    return None


def is_excluded_etf(name):
    name_u = str(name).upper()
    return any(str(k).upper() in name_u for k in EXCLUDE_KEYWORDS)


def nearest_business_date_str(days_back=10):
    d = datetime.now()
    for i in range(days_back + 1):
        s = (d - timedelta(days=i)).strftime("%Y%m%d")
        try:
            if stock.get_market_ticker_list(s, market="KOSPI"):
                return s
        except:
            pass
    return datetime.now().strftime("%Y%m%d")


# 6. Risk Layer — HAA + Fear & Greed + KOSPI

In [ ]:

CNN_FG_URL = "https://production.dataviz.cnn.io/index/fearandgreed/graphdata"

def get_haa_state():
    tickers = sorted(set(
        [CONFIG["haa_canary"]] +
        CONFIG["haa_offensive"] +
        CONFIG["haa_defensive"]
    ))

    close = download_close(tickers, period="2y")

    mom = pd.Series({
        t: momentum_13612u(close[t].dropna())
        for t in close.columns
    }).sort_values(ascending=False)

    tip_momentum = mom.get(CONFIG["haa_canary"], np.nan)

    return {
        "close": close,
        "momentum": mom,
        "tip_momentum": tip_momentum,
        "risk_on": bool(pd.notna(tip_momentum) and tip_momentum > 0),
    }


def get_cnn_fear_greed():
    if CONFIG["use_cnn_fear_greed"]:
        try:
            r = requests.get(
                CNN_FG_URL,
                headers={"User-Agent":"Mozilla/5.0"},
                timeout=10
            )
            r.raise_for_status()
            data = r.json()

            fg = data.get("fear_and_greed", {})

            for key in ["score","value","rating"]:
                v = fg.get(key)
                if isinstance(v, (int,float)) and 0 <= float(v) <= 100:
                    return {"value":float(v), "source":"CNN", "status":"ok"}
        except Exception as e:
            error = str(e)
        else:
            error = None
    else:
        error = None

    manual = CONFIG["manual_fear_greed"]

    if manual is not None:
        return {
            "value": float(manual),
            "source": "MANUAL",
            "status": "fallback"
        }

    return {
        "value": None,
        "source": None,
        "status": "missing",
        "error": error
    }


def get_kospi_close(period="1y"):
    df = yf.download(
        "^KS11",
        period=period,
        auto_adjust=True,
        progress=False
    )

    if isinstance(df.columns, pd.MultiIndex):
        s = df["Close"]
        if isinstance(s, pd.DataFrame):
            s = s.iloc[:,0]
    else:
        s = df["Close"]

    return s.dropna()


def tactical_cash_weight(risk_on, kospi, fear_greed=None):
    ma11 = kospi.rolling(11).mean()
    ma21 = kospi.rolling(21).mean()
    ret5 = kospi.pct_change(5)

    above11 = bool(kospi.iloc[-1] > ma11.iloc[-1])
    above21 = bool(kospi.iloc[-1] > ma21.iloc[-1])
    ret5_pos = bool(ret5.iloc[-1] > 0)

    reasons = []

    if not risk_on:
        cash = 0.30
        reasons.append("HAA Risk-Off")
    else:
        cash = 0.0
        reasons.append("HAA Risk-On")

        if not above11:
            cash = max(cash, 0.10)
            reasons.append("KOSPI < MA11")

        if not above21:
            cash = max(cash, 0.20)
            reasons.append("KOSPI < MA21")

        if (not above11) and (not above21) and (not ret5_pos):
            cash = 0.30
            reasons.append("KOSPI MA11/21 break + 5D weakness")

        if fear_greed is not None and pd.notna(fear_greed):
            fg = float(fear_greed)

            if fg >= CONFIG["fg_greed_level"]:
                cash = max(cash, 0.10)
                reasons.append("Fear&Greed greed")

            if fg >= CONFIG["fg_extreme_level"]:
                cash = max(cash, 0.20)
                reasons.append("Fear&Greed extreme")

            if (
                fg >= CONFIG["fg_max_risk_level"]
                and ((not above11) or (not ret5_pos))
            ):
                cash = max(cash, 0.30)
                reasons.append("Extreme greed + trend deterioration")

    return {
        "cash_weight": cash,
        "above11": above11,
        "above21": above21,
        "ret5_positive": ret5_pos,
        "fear_greed": fear_greed,
        "reasons": reasons,
    }


# 7. ETF Universe / Leader Sectors

In [ ]:

def get_all_kr_etfs():
    date = nearest_business_date_str()
    rows = []

    for t in stock.get_etf_ticker_list(date):
        try:
            name = stock.get_etf_ticker_name(t)
        except:
            name = ""

        rows.append({
            "ticker": str(t),
            "name": name,
            "sector": classify_sector_from_name(name),
            "excluded": is_excluded_etf(name),
        })

    return pd.DataFrame(rows)


def build_kr_etf_rank():
    universe = get_all_kr_etfs()

    universe = universe[
        (~universe["excluded"]) &
        (universe["sector"].notna())
    ].copy()

    date = nearest_business_date_str()

    try:
        snap = stock.get_etf_ohlcv_by_ticker(date)
        snap.index = snap.index.astype(str)
        universe = universe.merge(
            snap[["종가","거래대금"]],
            left_on="ticker",
            right_index=True,
            how="left"
        )
        universe = universe[
            universe["거래대금"].fillna(0) >= CONFIG["min_etf_trading_value"]
        ]
    except:
        universe["거래대금"] = np.nan

    yahoo_map = {f"{t}.KS":str(t) for t in universe["ticker"]}
    close = download_close(yahoo_map.keys(), period="1y").rename(columns=yahoo_map)

    meta = universe.set_index("ticker")[["name","sector","거래대금"]].to_dict("index")
    rows = []

    for ticker in close.columns:
        ticker = str(ticker)

        if ticker not in meta:
            continue

        s = close[ticker].dropna()

        if len(s) < 70:
            continue

        r = s.pct_change().dropna()
        neg = r.tail(63)
        neg = neg[neg < 0]

        downside = neg.std(ddof=1) * np.sqrt(252) if len(neg) > 1 else np.nan
        ret63 = return_n_days(s, 63)

        rows.append({
            "ticker": ticker,
            "name": meta[ticker]["name"],
            "sector": meta[ticker]["sector"],
            "거래대금": meta[ticker]["거래대금"],
            "ret21": return_n_days(s,21),
            "ret63": ret63,
            "ret126": return_n_days(s,126),
            "sortino63": ret63/downside if pd.notna(downside) and downside > 0 else np.nan,
            "above_ma11": bool(s.iloc[-1] > s.rolling(11).mean().iloc[-1]),
            "above_ma21": bool(s.iloc[-1] > s.rolling(21).mean().iloc[-1]),
        })

    rank = pd.DataFrame(rows)

    if rank.empty:
        return rank, close

    for src, dst in [
        ("ret21","rs21_pct"),
        ("ret63","rs63_pct"),
        ("ret126","rs126_pct"),
        ("sortino63","sortino_pct"),
    ]:
        rank[dst] = rank[src].rank(pct=True)

    rank["momentum_score"] = (
        0.20*rank["rs21_pct"] +
        0.50*rank["rs63_pct"] +
        0.15*rank["rs126_pct"] +
        0.15*rank["sortino_pct"]
    )

    rank["trend_score"] = (
        0.6*rank["above_ma11"].astype(int) +
        0.4*rank["above_ma21"].astype(int)
    )

    rank["kr_score"] = (
        0.80*rank["momentum_score"] +
        0.20*rank["trend_score"]
    )

    return rank.sort_values("kr_score", ascending=False).reset_index(drop=True), close


def build_us_sector_rank():
    tickers = sorted({t for x in US_SECTOR_ETFS.values() for t in x})
    close = download_close(tickers, period="1y")
    rows = []

    for sector, names in US_SECTOR_ETFS.items():
        tmp = []

        for t in names:
            if t not in close.columns:
                continue

            s = close[t].dropna()

            if len(s) < 70:
                continue

            ret63 = return_n_days(s,63)
            r = s.pct_change().dropna().tail(63)
            neg = r[r < 0]
            dd = neg.std(ddof=1)*np.sqrt(252) if len(neg)>1 else np.nan

            tmp.append({
                "ret63": ret63,
                "sortino": ret63/dd if pd.notna(dd) and dd>0 else np.nan,
                "above11": s.iloc[-1] > s.rolling(11).mean().iloc[-1],
                "above21": s.iloc[-1] > s.rolling(21).mean().iloc[-1],
            })

        if not tmp:
            continue

        x = pd.DataFrame(tmp)

        rows.append({
            "sector": sector,
            "us_ret63": x["ret63"].mean(),
            "us_sortino": x["sortino"].mean(),
            "us_above11": bool(x["above11"].mean() >= 0.5),
            "us_above21": bool(x["above21"].mean() >= 0.5),
        })

    rank = pd.DataFrame(rows)

    if rank.empty:
        return rank

    rank["us_rs63_pct"] = rank["us_ret63"].rank(pct=True)
    rank["us_sortino_pct"] = rank["us_sortino"].rank(pct=True)

    rank["us_score"] = (
        0.60*rank["us_rs63_pct"] +
        0.20*rank["us_sortino_pct"] +
        0.12*rank["us_above11"].astype(int) +
        0.08*rank["us_above21"].astype(int)
    )

    return rank.sort_values("us_score", ascending=False).reset_index(drop=True)


def build_leaders_today():
    kr_rank, kr_close = build_kr_etf_rank()
    us_rank = build_us_sector_rank()

    if kr_rank.empty:
        return pd.DataFrame(), kr_rank, us_rank, kr_close

    reps = (
        kr_rank.sort_values("kr_score", ascending=False)
        .groupby("sector", as_index=False)
        .first()
        [["sector","ticker","name","kr_score","ret63","above_ma11","above_ma21"]]
        .rename(columns={
            "ticker":"rep_ticker",
            "name":"rep_name",
            "ret63":"kr_ret63"
        })
    )

    avgs = (
        kr_rank.groupby("sector", as_index=False)
        .agg(
            kr_sector_avg_score=("kr_score","mean"),
            kr_etf_count=("ticker","count")
        )
    )

    out = reps.merge(avgs, on="sector", how="left")

    if not us_rank.empty:
        out = out.merge(
            us_rank[["sector","us_score","us_ret63","us_above11","us_above21"]],
            on="sector",
            how="left"
        )

    out["us_score"] = out.get("us_score",0)
    out["us_score"] = out["us_score"].fillna(0)

    out["us_above11"] = out.get("us_above11",False)
    out["us_above11"] = out["us_above11"].fillna(False)

    out["cross_market_confirm"] = (
        out["us_above11"] &
        (out["us_score"] >= 0.60)
    )

    out["final_sector_score"] = (
        0.70*out["kr_score"] +
        0.15*out["kr_sector_avg_score"] +
        0.15*out["cross_market_confirm"].astype(int)
    )

    leaders = (
        out[out["above_ma11"]]
        .sort_values("final_sector_score", ascending=False)
        .head(CONFIG["leader_sector_count"])
        .reset_index(drop=True)
    )

    return leaders, kr_rank, us_rank, kr_close


# 8. ETF Snapshot / Alpha

In [ ]:

ETF_CONSTITUENT_DIR = BASE_DIR / "etf_constituents"

def get_etf_constituents(etf_ticker, date=None):
    if date is None:
        date = nearest_business_date_str()

    try:
        df = stock.get_etf_portfolio_deposit_file(date, str(etf_ticker))
    except:
        return pd.DataFrame()

    if df is None or len(df) == 0:
        return pd.DataFrame()

    out = df.copy()
    out.index = out.index.astype(str)
    out = out[out.index.str.fullmatch(r"\d{6}", na=False)].copy()
    out["ticker"] = out.index

    if "구성종목명" not in out.columns:
        out["구성종목명"] = out["ticker"].apply(stock.get_market_ticker_name)

    return out.reset_index(drop=True)


def save_etf_snapshot(sector, etf_ticker, etf_name=None):
    date = nearest_business_date_str()
    df = get_etf_constituents(etf_ticker, date)

    if df.empty:
        return None

    date_tag = pd.Timestamp(date).strftime("%Y-%m-%d")

    df["sector"] = sector
    df["source_etf"] = str(etf_ticker)
    df["source_etf_name"] = etf_name
    df["snapshot_date"] = date_tag

    folder = ETF_CONSTITUENT_DIR / sector / str(etf_ticker)
    folder.mkdir(parents=True, exist_ok=True)

    path = folder / f"{date_tag}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")

    return path


def get_latest_close_krx(ticker):
    end = datetime.now()

    for i in range(10):
        d = (end - timedelta(days=i)).strftime("%Y%m%d")

        try:
            df = stock.get_market_ohlcv_by_ticker(d, market="ALL")
            if str(ticker) in df.index.astype(str):
                df.index = df.index.astype(str)
                return float(df.loc[str(ticker),"종가"])
        except:
            pass

        try:
            df = stock.get_etf_ohlcv_by_ticker(d)
            df.index = df.index.astype(str)
            if str(ticker) in df.index:
                return float(df.loc[str(ticker),"종가"])
        except:
            pass

    return np.nan


def get_stock_features_today(ticker):
    end = datetime.now()
    start = end - timedelta(days=500)

    try:
        px = stock.get_market_ohlcv_by_date(
            start.strftime("%Y%m%d"),
            end.strftime("%Y%m%d"),
            ticker
        )
        flow = stock.get_market_trading_value_by_date(
            start.strftime("%Y%m%d"),
            end.strftime("%Y%m%d"),
            ticker,
            on="순매수"
        )
    except:
        return None

    if px.empty or flow.empty:
        return None

    mcap = np.nan

    for i in range(10):
        ds = (end - timedelta(days=i)).strftime("%Y%m%d")
        try:
            mc = stock.get_market_cap_by_ticker(ds)
            mc.index = mc.index.astype(str)
            if ticker in mc.index:
                mcap = float(mc.loc[ticker,"시가총액"])
                break
        except:
            pass

    if not np.isfinite(mcap) or mcap <= 0:
        return None

    foreign_col = find_col(flow,"외국인")
    inst_col = find_col(flow,"기관")

    if foreign_col is None and inst_col is None:
        return None

    close = px["종가"].astype(float)
    high = px["고가"].astype(float)
    value = px["거래대금"].astype(float)

    f = flow[foreign_col].astype(float) if foreign_col else pd.Series(0.0,index=flow.index)
    i = flow[inst_col].astype(float) if inst_col else pd.Series(0.0,index=flow.index)

    feat = {
        "ticker": ticker,
        "name": stock.get_market_ticker_name(ticker),
        "mcap": mcap,
    }

    for label, s in [("foreign",f),("institution",i)]:
        for n in [5,20,60]:
            raw = float(s.tail(n).sum()) if len(s)>=n else np.nan
            feat[f"{label}_{n}_to_mcap"] = raw/mcap if pd.notna(raw) else np.nan

    combined = f.add(i, fill_value=0)
    roll20 = combined.rolling(20).sum()/mcap
    hist20 = roll20.dropna()
    current = hist20.iloc[-1] if len(hist20) else np.nan

    feat["flow_hist_percentile"] = (
        float((hist20 <= current).mean())
        if len(hist20)>=20 and pd.notna(current)
        else np.nan
    )

    for n in [5,20,60,120]:
        feat[f"ret{n}"] = return_n_days(close,n)

    ma20 = close.rolling(20).mean()
    ma60 = close.rolling(60).mean()
    ma120 = close.rolling(120).mean()

    feat["trend20_60"] = bool(
        len(close)>=60 and close.iloc[-1] > ma20.iloc[-1] > ma60.iloc[-1]
    )
    feat["trend60_120"] = bool(
        len(close)>=120 and ma60.iloc[-1] > ma120.iloc[-1]
    )

    feat["avg_value_20"] = value.tail(20).mean()
    avg5 = value.tail(5).mean()
    feat["value_expansion"] = avg5/feat["avg_value_20"] if feat["avg_value_20"]>0 else np.nan

    high252 = high.tail(252).max()
    feat["dist_to_52w_high"] = close.iloc[-1]/high252 - 1 if high252>0 else np.nan

    dist_ma20 = close.iloc[-1]/ma20.iloc[-1]-1 if pd.notna(ma20.iloc[-1]) else np.nan

    pullback = 0

    if feat["trend20_60"]:
        pullback += 30

        if pd.notna(dist_ma20):
            if -0.03 <= dist_ma20 <= 0.03:
                pullback += 30
            elif -0.06 <= dist_ma20 <= 0.06:
                pullback += 15

        if pd.notna(feat["ret5"]):
            if -0.08 <= feat["ret5"] <= 0:
                pullback += 25
            elif 0 < feat["ret5"] <= 0.03:
                pullback += 10

        if pd.notna(feat["value_expansion"]) and feat["value_expansion"] < 1:
            pullback += 15

    feat["pullback_score"] = pullback

    row = px.iloc[-1]
    prev = px.iloc[-61:-1] if len(px)>=61 else pd.DataFrame()

    if not prev.empty:
        o,h,c = float(row["시가"]),float(row["고가"]),float(row["종가"])
        prev_high = float(prev["고가"].max())
        body = abs(c-o)
        upper = h-max(o,c)

        feat["reversal_risk"] = bool(
            h>=prev_high and c<o and upper/max(body,1e-9)>=1.0
        )
    else:
        feat["reversal_risk"] = False

    return feat


def score_alpha(df):
    if df.empty:
        return df

    x = df.copy()

    cross = pd.Series(0.0,index=x.index)

    for investor, inv_w in CONFIG["investor_weights"].items():
        inv_score = pd.Series(0.0,index=x.index)

        for n, win_w in CONFIG["flow_window_weights"].items():
            col = f"{investor}_{n}_to_mcap"

            if col in x.columns:
                inv_score += (1-safe_pct_rank(x[col])).fillna(0)*100*win_w

        x[f"{investor}_empty_score"] = inv_score
        cross += inv_score*inv_w

    x["empty_cross_score"] = cross
    x["empty_hist_score"] = (1-x["flow_hist_percentile"])*100

    x["supply_empty_score"] = (
        0.65*x["empty_cross_score"] +
        0.35*x["empty_hist_score"].fillna(50)
    )

    x["price_momentum_score"] = (
        0.25*safe_pct_rank(x["ret20"]) +
        0.45*safe_pct_rank(x["ret60"]) +
        0.30*safe_pct_rank(x["ret120"])
    )*100

    x["trend_score"] = (
        60*x["trend20_60"].astype(int) +
        40*x["trend60_120"].astype(int)
    )

    x["liquidity_score"] = safe_pct_rank(x["value_expansion"])*100

    def high_score(v):
        if pd.isna(v): return 0
        if -0.05 <= v <= 0: return 100
        if -0.10 <= v < -0.05: return 80
        if -0.20 <= v < -0.10: return 60
        if -0.30 <= v < -0.20: return 30
        return 10

    x["high_proximity_score"] = x["dist_to_52w_high"].apply(high_score)

    x["heat_penalty"] = 0.0
    x.loc[x["ret20"]>0.12,"heat_penalty"] = 7
    x.loc[x["ret20"]>0.20,"heat_penalty"] = 15
    x.loc[x["ret20"]>0.30,"heat_penalty"] = 25

    w = CONFIG["alpha_weights"]

    x["alpha_score"] = (
        w["supply_empty"]*x["supply_empty_score"] +
        w["price_momentum"]*x["price_momentum_score"] +
        w["trend"]*x["trend_score"] +
        w["liquidity"]*x["liquidity_score"] +
        w["high_proximity"]*x["high_proximity_score"] +
        w["pullback"]*x["pullback_score"] -
        x["heat_penalty"] -
        x["reversal_risk"].astype(int)*35
    )

    x["eligible_alpha"] = (
        x["trend20_60"] &
        (x["avg_value_20"].fillna(0) >= CONFIG["min_stock_trading_value"])
    )

    return x.sort_values(
        ["eligible_alpha","alpha_score"],
        ascending=[False,False]
    ).reset_index(drop=True)


def build_alpha_candidates(leaders):
    all_rows = []

    for _, r in leaders.iterrows():
        sector = r["sector"]
        etf_ticker = str(r["rep_ticker"])

        pdf = get_etf_constituents(etf_ticker)

        if pdf.empty:
            continue

        if "비중" in pdf.columns:
            pdf = pdf.sort_values("비중",ascending=False)

        feats = []

        for t in pdf["ticker"].head(CONFIG["max_constituents_per_etf"]):
            feat = get_stock_features_today(str(t))
            if feat is not None:
                feats.append(feat)

        if not feats:
            continue

        scored = score_alpha(pd.DataFrame(feats))
        top = scored[scored["eligible_alpha"]].head(CONFIG["alpha_top_n_per_sector"]).copy()

        if not top.empty:
            top["sector"] = sector
            top["source_etf"] = etf_ticker
            all_rows.append(top)

    return pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()


# 9. Portfolio / Execution

In [ ]:

def load_holdings():
    path = BASE_DIR / "portfolio" / "current_holdings.csv"

    cols = [
        "ticker","name","sector","asset_type",
        "quantity","avg_price","current_price"
    ]

    if not path.exists():
        pd.DataFrame(columns=cols).to_csv(path,index=False,encoding="utf-8-sig")

    try:
        df = pd.read_csv(path)
    except:
        return pd.DataFrame(columns=cols), path

    for c in cols:
        if c not in df.columns:
            df[c] = np.nan

    for c in ["quantity","avg_price","current_price"]:
        df[c] = pd.to_numeric(df[c],errors="coerce").fillna(0)

    df["market_value"] = df["quantity"]*df["current_price"]

    return df, path


def refresh_holdings_prices(holdings):
    if holdings.empty:
        return holdings

    out = holdings.copy()

    for idx, r in out.iterrows():
        p = get_latest_close_krx(str(r["ticker"]))
        if pd.notna(p) and p>0:
            out.loc[idx,"current_price"] = p

    out["market_value"] = out["quantity"]*out["current_price"]
    return out


def build_target_portfolio(leaders, alpha, stock_exposure):
    rows = []

    core_total = stock_exposure*CONFIG["etf_core_within_stock"]

    if not leaders.empty:
        etf_w = core_total/len(leaders)

        for _,r in leaders.iterrows():
            rows.append({
                "ticker":str(r["rep_ticker"]),
                "name":r["rep_name"],
                "sector":r["sector"],
                "asset_type":"ETF",
                "target_weight":etf_w,
                "alpha_score":np.nan,
            })

    if not alpha.empty:
        alpha_total = stock_exposure*CONFIG["alpha_within_stock"]
        eq_w = min(
            alpha_total/len(alpha),
            CONFIG["max_single_stock_weight"]
        )

        for _,r in alpha.iterrows():
            rows.append({
                "ticker":str(r["ticker"]),
                "name":r["name"],
                "sector":r["sector"],
                "asset_type":"STOCK",
                "target_weight":eq_w,
                "alpha_score":r["alpha_score"],
            })

    target = pd.DataFrame(rows)

    if target.empty:
        return target

    # sector cap
    sector_sum = target.groupby("sector")["target_weight"].sum()

    for sector,total in sector_sum.items():
        if total > CONFIG["max_sector_weight"]:
            mask = target["sector"]==sector
            target.loc[mask,"target_weight"] *= CONFIG["max_sector_weight"]/total

    return target


def build_rebalance_table(holdings,target,total_equity):
    current = holdings.copy()

    if current.empty:
        current = pd.DataFrame(columns=[
            "ticker","name","sector","asset_type",
            "quantity","current_price","market_value"
        ])

    current["current_weight"] = (
        current["market_value"]/total_equity
        if total_equity>0 else 0
    )

    merged = current.merge(
        target,
        on="ticker",
        how="outer",
        suffixes=("_current","_target")
    )

    for c in ["name","sector","asset_type"]:
        merged[c] = merged.get(f"{c}_target")
        if f"{c}_current" in merged.columns:
            merged[c] = merged[c].fillna(merged[f"{c}_current"])

    for c in ["quantity","market_value","current_weight","target_weight"]:
        merged[c] = pd.to_numeric(merged.get(c,0),errors="coerce").fillna(0)

    price_map = {}

    for _,r in current.iterrows():
        if r.get("current_price",0)>0:
            price_map[str(r["ticker"])] = float(r["current_price"])

    for t in merged["ticker"].astype(str):
        if t not in price_map:
            price_map[t] = get_latest_close_krx(t)

    merged["price"] = merged["ticker"].astype(str).map(price_map)
    merged["target_value"] = merged["target_weight"]*total_equity
    merged["trade_value"] = merged["target_value"]-merged["market_value"]
    merged["weight_gap"] = merged["target_weight"]-merged["current_weight"]

    def classify(r):
        if r["target_weight"]<=0 and r["current_weight"]>0:
            return "SELL"
        if r["current_weight"]<=0 and r["target_weight"]>0:
            return "BUY"
        if abs(r["weight_gap"])<=0.005:
            return "HOLD"
        return "BUY_MORE" if r["trade_value"]>0 else "REDUCE"

    merged["rebalance_action"] = merged.apply(classify,axis=1)

    return merged


ACTION_PRIORITY = {
    "SELL":1,
    "ROTATE":2,
    "REDUCE":3,
    "BUY":4,
    "BUY_MORE":5,
    "HOLD":99,
}


def approve_orders(rebalance,holdings,current_cash,total_equity,target_cash_weight):
    if rebalance.empty or total_equity<=0:
        return pd.DataFrame()

    x = rebalance.copy()
    x["priority"] = x["rebalance_action"].map(ACTION_PRIORITY).fillna(50)

    if "alpha_score" not in x.columns:
        x["alpha_score"] = np.nan

    x = x.sort_values(
        ["priority","alpha_score","trade_value"],
        ascending=[True,False,False]
    )

    turnover_budget = total_equity*CONFIG["max_daily_turnover"]
    used_turnover = 0.0

    available_cash = float(current_cash)
    min_cash_allowed = total_equity*max(
        target_cash_weight-CONFIG["cash_tolerance"],0
    )

    sector_weights = (
        holdings.groupby("sector")["market_value"].sum()/total_equity
    ).to_dict() if not holdings.empty else {}

    approved = []
    new_buy_count = 0

    for side_group in [["SELL","ROTATE","REDUCE"],["BUY","BUY_MORE"]]:
        for _,r in x.iterrows():
            action = r["rebalance_action"]

            if action not in side_group:
                continue

            price = r.get("price",np.nan)
            desired = abs(float(r.get("trade_value",0)))

            if not pd.notna(price) or price<=0 or desired<CONFIG["min_trade_value"]:
                continue

            remaining = turnover_budget-used_turnover

            if remaining<=0:
                break

            sector = r.get("sector",None)

            if action in ["SELL","ROTATE","REDUCE"]:
                qty = int(r.get("quantity",0))
                approve_value = min(desired,remaining)
                order_qty = min(qty,math.floor(approve_value/price))

                if order_qty<=0:
                    continue

                actual = order_qty*price
                signed_qty = -order_qty
                signed_value = -actual

                available_cash += actual

                if sector in sector_weights:
                    sector_weights[sector] = max(
                        0,
                        sector_weights[sector]-actual/total_equity
                    )

            else:
                if action=="BUY" and new_buy_count>=CONFIG["max_new_buys"]:
                    continue

                spendable = max(0,available_cash-min_cash_allowed)

                sector_room = max(
                    0,
                    (CONFIG["max_sector_weight"]-sector_weights.get(sector,0))*total_equity
                )

                approve_value = min(desired,remaining,spendable,sector_room)

                if approve_value<CONFIG["min_trade_value"]:
                    continue

                order_qty = math.floor(approve_value/price)

                if order_qty<=0:
                    continue

                actual = order_qty*price
                signed_qty = order_qty
                signed_value = actual

                available_cash -= actual
                sector_weights[sector] = sector_weights.get(sector,0)+actual/total_equity

                if action=="BUY":
                    new_buy_count += 1

            out = r.copy()
            out["approved_order_qty"] = signed_qty
            out["approved_trade_value"] = signed_value
            out["approved"] = True
            approved.append(out)

            used_turnover += abs(actual)

    if not approved:
        return pd.DataFrame()

    out = pd.DataFrame(approved)
    out["approved_turnover"] = used_turnover/total_equity
    out["post_cash_estimate"] = available_cash

    return out


# 10. NAV / Reporting

In [ ]:

def append_nav(holdings,current_cash,haa_state,target_cash_weight):
    path = BASE_DIR/"portfolio"/"nav_history.csv"
    today = datetime.now().strftime("%Y-%m-%d")

    stock_value = holdings["market_value"].sum() if not holdings.empty else 0

    etf_value = (
        holdings.loc[
            holdings["asset_type"].astype(str).str.upper()=="ETF",
            "market_value"
        ].sum()
        if not holdings.empty else 0
    )

    alpha_value = (
        holdings.loc[
            holdings["asset_type"].astype(str).str.upper()=="STOCK",
            "market_value"
        ].sum()
        if not holdings.empty else 0
    )

    row = pd.DataFrame([{
        "date":today,
        "total_equity":current_cash+stock_value,
        "cash":current_cash,
        "stock_value":stock_value,
        "etf_value":etf_value,
        "alpha_value":alpha_value,
        "haa_state":haa_state,
        "target_cash_weight":target_cash_weight,
    }])

    if path.exists():
        old = pd.read_csv(path)
        old = old[old["date"].astype(str)!=today]
        out = pd.concat([old,row],ignore_index=True)
    else:
        out = row

    out.sort_values("date").to_csv(path,index=False,encoding="utf-8-sig")
    return out.sort_values("date").reset_index(drop=True)


def nav_stats(nav):
    if len(nav)<2:
        return pd.Series(dtype=float),pd.DataFrame()

    x = nav.copy()
    x["date"] = pd.to_datetime(x["date"])
    x = x.sort_values("date").set_index("date")

    x["daily_return"] = x["total_equity"].pct_change()
    x["cum_return"] = x["total_equity"]/x["total_equity"].iloc[0]-1
    peak = x["total_equity"].cummax()
    x["drawdown"] = x["total_equity"]/peak-1

    r = x["daily_return"].dropna()
    vol = r.std(ddof=1)*np.sqrt(252) if len(r)>1 else np.nan
    sharpe = r.mean()*252/vol if pd.notna(vol) and vol>0 else np.nan

    days = max((x.index[-1]-x.index[0]).days,1)
    years = days/365.25
    cagr = (
        (x["total_equity"].iloc[-1]/x["total_equity"].iloc[0])**(1/years)-1
        if years>0 and x["total_equity"].iloc[0]>0
        else np.nan
    )

    s = pd.Series({
        "LatestNAV":x["total_equity"].iloc[-1],
        "TotalReturn":x["cum_return"].iloc[-1],
        "CAGR":cagr,
        "Volatility":vol,
        "Sharpe":sharpe,
        "MaxDrawdown":x["drawdown"].min(),
    })

    return s,x


def build_daily_report(result):
    rows = []

    rows += [
        {"section":"RISK","item":"HAA","value":"RISK-ON" if result["haa"]["risk_on"] else "RISK-OFF"},
        {"section":"RISK","item":"TIP 13612U","value":result["haa"]["tip_momentum"]},
        {"section":"RISK","item":"FearGreed","value":result["fear_greed"]["value"]},
        {"section":"ALLOCATION","item":"CashWeight","value":result["cash_info"]["cash_weight"]},
        {"section":"ALLOCATION","item":"StockExposure","value":1-result["cash_info"]["cash_weight"]},
        {"section":"SIGNAL","item":"LeaderCount","value":len(result["leaders"])},
        {"section":"SIGNAL","item":"AlphaCount","value":len(result["alpha_candidates"])},
        {"section":"EXECUTION","item":"ApprovedOrders","value":len(result["approved_orders"])},
    ]

    for k,v in result["nav_stats"].items():
        rows.append({"section":"PERFORMANCE","item":k,"value":v})

    return pd.DataFrame(rows)


# 11. DAILY ENTRYPOINT

In [ ]:

def run_daily():
    print("=== HAA Leader System v2.0 / DAILY ===")

    # 1. Risk
    haa = get_haa_state()
    fg = get_cnn_fear_greed()
    kospi = get_kospi_close()

    cash_info = tactical_cash_weight(
        haa["risk_on"],
        kospi,
        fg["value"]
    )

    stock_exposure = 1-cash_info["cash_weight"]

    # 2. Leaders
    leaders, kr_rank, us_rank, kr_close = build_leaders_today()

    # 3. Alpha
    alpha = build_alpha_candidates(leaders)

    # 4. Save ETF snapshots
    saved_snapshots = []

    for _,r in leaders.iterrows():
        p = save_etf_snapshot(
            r["sector"],
            str(r["rep_ticker"]),
            r["rep_name"]
        )
        if p is not None:
            saved_snapshots.append(str(p))

    # 5. Holdings
    holdings, holdings_path = load_holdings()
    holdings = refresh_holdings_prices(holdings)

    holdings.to_csv(
        holdings_path,
        index=False,
        encoding="utf-8-sig"
    )

    stock_value = holdings["market_value"].sum() if not holdings.empty else 0
    total_equity = CONFIG["current_cash"]+stock_value

    # 6. Target / Rebalance / Orders
    target = build_target_portfolio(
        leaders,
        alpha,
        stock_exposure
    )

    rebalance = (
        build_rebalance_table(holdings,target,total_equity)
        if total_equity>0
        else pd.DataFrame()
    )

    approved = (
        approve_orders(
            rebalance,
            holdings,
            CONFIG["current_cash"],
            total_equity,
            cash_info["cash_weight"]
        )
        if total_equity>0
        else pd.DataFrame()
    )

    # 7. NAV
    nav = append_nav(
        holdings,
        CONFIG["current_cash"],
        "RISK-ON" if haa["risk_on"] else "RISK-OFF",
        cash_info["cash_weight"]
    )

    stats, nav_series = nav_stats(nav)

    result = {
        "haa":haa,
        "fear_greed":fg,
        "cash_info":cash_info,
        "leaders":leaders,
        "kr_rank":kr_rank,
        "us_rank":us_rank,
        "alpha_candidates":alpha,
        "holdings":holdings,
        "target":target,
        "rebalance":rebalance,
        "approved_orders":approved,
        "nav_history":nav,
        "nav_stats":stats,
        "nav_series":nav_series,
        "saved_snapshots":saved_snapshots,
        "base_dir":BASE_DIR,
    }

    report = build_daily_report(result)
    result["report"] = report

    # 8. Save
    today = datetime.now().strftime("%Y-%m-%d")

    leaders.to_csv(
        BASE_DIR/"snapshots"/f"{today}_leaders_v20.csv",
        index=False,encoding="utf-8-sig"
    )

    if not alpha.empty:
        alpha.to_csv(
            BASE_DIR/"snapshots"/f"{today}_alpha_v20.csv",
            index=False,encoding="utf-8-sig"
        )

    if not target.empty:
        target.to_csv(
            BASE_DIR/"snapshots"/f"{today}_target_v20.csv",
            index=False,encoding="utf-8-sig"
        )

    if not approved.empty:
        approved.to_csv(
            BASE_DIR/"rebalancing"/f"{today}_approved_orders_v20.csv",
            index=False,encoding="utf-8-sig"
        )

    report.to_csv(
        BASE_DIR/"reports"/f"{today}_daily_report_v20.csv",
        index=False,encoding="utf-8-sig"
    )

    # 9. Display
    display(report)

    print("\n[Leaders]")
    display(
        leaders[
            ["sector","rep_ticker","rep_name",
             "cross_market_confirm","final_sector_score"]
        ]
        if not leaders.empty else leaders
    )

    print("\n[Alpha]")
    display(
        alpha[
            ["sector","ticker","name","alpha_score",
             "supply_empty_score","price_momentum_score",
             "reversal_risk"]
        ]
        if not alpha.empty else alpha
    )

    print("\n[Approved Orders]")
    display(
        approved[
            ["ticker","name","sector","rebalance_action",
             "price","approved_order_qty","approved_trade_value"]
        ]
        if not approved.empty else approved
    )

    return result


# 12. RESEARCH ENTRYPOINT

In [ ]:

def rolling_13612u_series(close_series):
    s = close_series.dropna()
    out = pd.Series(index=s.index,dtype=float)

    for d in s.index:
        hist = s.loc[:d]
        vals = []

        for n in [21,63,126,252]:
            if len(hist)>n:
                vals.append(hist.iloc[-1]/hist.iloc[-(n+1)]-1)

        if vals:
            out.loc[d] = np.mean(vals)

    return out


def build_monthly_haa_regime(tip_close):
    mom = rolling_13612u_series(tip_close)
    rows = []

    for month_end in tip_close.resample("M").last().dropna().index:
        dates = tip_close.index[tip_close.index<=month_end]

        if len(dates)==0:
            continue

        signal_date = dates[-1]

        if signal_date not in mom.index or pd.isna(mom.loc[signal_date]):
            continue

        loc = tip_close.index.get_loc(signal_date)

        if loc+1>=len(tip_close.index):
            continue

        rows.append({
            "signal_date":signal_date,
            "apply_date":tip_close.index[loc+1],
            "risk_on":bool(mom.loc[signal_date]>0),
            "tip_13612u":mom.loc[signal_date],
        })

    return pd.DataFrame(rows)


def regime_log_to_daily(index,regime_log):
    s = pd.Series(False,index=index,dtype=bool)
    current = False

    apply_map = {
        r["apply_date"]:bool(r["risk_on"])
        for _,r in regime_log.iterrows()
        if r["apply_date"] in index
    }

    for d in index:
        if d in apply_map:
            current = apply_map[d]
        s.loc[d] = current

    return s


def performance_stats(result):
    if result is None or result.empty:
        return pd.Series(dtype=float)

    r = result["net_ret"].dropna()
    eq = result["equity"].dropna()

    if len(r)<2 or len(eq)<2:
        return pd.Series(dtype=float)

    years = max(len(r)/252,1/252)
    cagr = eq.iloc[-1]**(1/years)-1 if eq.iloc[-1]>0 else np.nan
    vol = r.std(ddof=1)*np.sqrt(252)
    sharpe = r.mean()*252/vol if vol>0 else np.nan
    dd = eq/eq.cummax()-1

    return pd.Series({
        "CAGR":cagr,
        "Volatility":vol,
        "Sharpe":sharpe,
        "MaxDrawdown":dd.min(),
        "TotalReturn":eq.iloc[-1]-1,
        "TurnoverPerDay":result.get("turnover",pd.Series(0,index=result.index)).mean(),
    })


def run_research():
    print("=== HAA Leader System v2.0 / RESEARCH ===")

    # 현재 연구 진입점은 v1.7까지 만든 구조를 연결하기 위한 준비 루틴
    # 장기 ETF 가격패널
    meta, kr_close = build_long_kr_etf_history()
    us_close = build_us_long_history()

    # HAA history
    haa_tickers = sorted(set(
        [CONFIG["haa_canary"]] +
        CONFIG["haa_offensive"] +
        CONFIG["haa_defensive"]
    ))

    haa_close = download_close(haa_tickers,period="10y")
    tip = haa_close[CONFIG["haa_canary"]].dropna()
    haa_log = build_monthly_haa_regime(tip)
    haa_daily = regime_log_to_daily(kr_close.index,haa_log)

    result = {
        "kr_meta":meta,
        "kr_close":kr_close,
        "us_close":us_close,
        "haa_log":haa_log,
        "haa_daily":haa_daily,
    }

    print("KR ETF universe:",len(meta))
    print("KR history days:",len(kr_close))
    print("HAA regime rows:",len(haa_log))

    print(
        "\n다음 셀의 v1.6/v1.7 연구 함수들을 이용해 "
        "leader_history, Alpha snapshot, Walk-forward OOS를 실행할 수 있습니다."
    )

    return result



# 13. 연구용 함수 호환성

v2.0에서는 실전 코드를 깔끔하게 정리했지만,
v1.6~v1.7의 과거 주도섹터/Walk-forward 연구 함수는 그대로 재사용할 수 있습니다.

아래 셀은 이전 노트북의 연구 함수를 가져오는 대신,
필요한 핵심 함수만 재정의합니다.


In [ ]:

def make_walk_forward_windows(
    index,
    train_months=WF_TRAIN_MONTHS,
    test_months=WF_TEST_MONTHS,
):
    idx = pd.DatetimeIndex(index).sort_values().unique()

    if len(idx) == 0:
        return []

    start = idx.min().normalize()
    end = idx.max().normalize()

    windows = []
    train_start = start

    while True:
        train_end = train_start + pd.DateOffset(months=train_months) - pd.Timedelta(days=1)
        test_start = train_end + pd.Timedelta(days=1)
        test_end = test_start + pd.DateOffset(months=test_months) - pd.Timedelta(days=1)

        if test_start > end:
            break

        windows.append({
            "train_start": train_start,
            "train_end": min(train_end, end),
            "test_start": test_start,
            "test_end": min(test_end, end),
        })

        train_start = train_start + pd.DateOffset(months=test_months)

        if train_start >= end:
            break

    return windows


NameError: name 'WF_TRAIN_MONTHS' is not defined

In [ ]:

HISTORICAL_ETF_PERIOD = "10y"

def build_long_kr_etf_history():
    universe = get_all_kr_etfs()
    universe = universe[
        (~universe["excluded"]) &
        (universe["sector"].notna())
    ].copy()

    yahoo_map = {f"{t}.KS": str(t) for t in universe["ticker"].astype(str)}
    close = download_close(yahoo_map.keys(), period=HISTORICAL_ETF_PERIOD)
    close = close.rename(columns=yahoo_map)

    available = set(close.columns.astype(str))
    meta = universe[universe["ticker"].astype(str).isin(available)].copy()

    return meta, close


def build_us_long_history():
    tickers = sorted({t for lst in US_SECTOR_ETFS.values() for t in lst})
    return download_close(tickers, period=HISTORICAL_ETF_PERIOD)


In [ ]:

def build_kr_etf_rank_asof(close_panel, meta_df, asof_date, min_history=70):
    asof = pd.Timestamp(asof_date)
    px = close_panel.loc[close_panel.index <= asof].copy()

    if px.empty:
        return pd.DataFrame()

    meta = meta_df.set_index("ticker")[["name","sector"]].to_dict("index")
    rows = []

    for ticker in px.columns:
        ticker = str(ticker)
        if ticker not in meta:
            continue

        s = px[ticker].dropna()
        if len(s) < min_history:
            continue

        last_date = pd.Timestamp(s.index[-1])
        if (asof - last_date).days > 10:
            continue

        r = s.pct_change().dropna()
        ret21 = return_n_days(s, 21)
        ret63 = return_n_days(s, 63)
        ret126 = return_n_days(s, 126)

        neg = r.tail(63)
        neg = neg[neg < 0]
        downside = neg.std(ddof=1) * np.sqrt(252) if len(neg) > 1 else np.nan
        sortino = ret63 / downside if pd.notna(downside) and downside > 0 else np.nan

        rows.append({
            "ticker": ticker,
            "name": meta[ticker]["name"],
            "sector": meta[ticker]["sector"],
            "ret21": ret21,
            "ret63": ret63,
            "ret126": ret126,
            "sortino63": sortino,
            "above_ma11": bool(s.iloc[-1] > s.rolling(11).mean().iloc[-1]),
            "above_ma21": bool(s.iloc[-1] > s.rolling(21).mean().iloc[-1]),
        })

    rank = pd.DataFrame(rows)
    if rank.empty:
        return rank

    rank["rs21_pct"] = rank["ret21"].rank(pct=True)
    rank["rs63_pct"] = rank["ret63"].rank(pct=True)
    rank["rs126_pct"] = rank["ret126"].rank(pct=True)
    rank["sortino_pct"] = rank["sortino63"].rank(pct=True)

    rank["momentum_score"] = (
        0.20 * rank["rs21_pct"] +
        0.50 * rank["rs63_pct"] +
        0.15 * rank["rs126_pct"] +
        0.15 * rank["sortino_pct"]
    )

    rank["trend_score"] = (
        0.6 * rank["above_ma11"].astype(int) +
        0.4 * rank["above_ma21"].astype(int)
    )

    rank["kr_score"] = 0.80 * rank["momentum_score"] + 0.20 * rank["trend_score"]
    return rank.sort_values("kr_score", ascending=False).reset_index(drop=True)


In [ ]:

def build_us_sector_rank_asof(close_panel, asof_date, min_history=70):
    asof = pd.Timestamp(asof_date)
    px = close_panel.loc[close_panel.index <= asof].copy()

    rows = []

    for sector, tickers in US_SECTOR_ETFS.items():
        tmp = []

        for t in tickers:
            if t not in px.columns:
                continue

            s = px[t].dropna()
            if len(s) < min_history:
                continue

            if (asof - pd.Timestamp(s.index[-1])).days > 10:
                continue

            r = s.pct_change().dropna()
            ret63 = return_n_days(s, 63)

            neg = r.tail(63)
            neg = neg[neg < 0]
            downside = neg.std(ddof=1) * np.sqrt(252) if len(neg) > 1 else np.nan
            sortino = ret63 / downside if pd.notna(downside) and downside > 0 else np.nan

            tmp.append({
                "ret63": ret63,
                "sortino": sortino,
                "above11": bool(s.iloc[-1] > s.rolling(11).mean().iloc[-1]),
                "above21": bool(s.iloc[-1] > s.rolling(21).mean().iloc[-1]),
            })

        if not tmp:
            continue

        x = pd.DataFrame(tmp)
        rows.append({
            "sector": sector,
            "us_ret63": x["ret63"].mean(),
            "us_sortino": x["sortino"].mean(),
            "us_above11": bool(x["above11"].mean() >= 0.5),
            "us_above21": bool(x["above21"].mean() >= 0.5),
        })

    rank = pd.DataFrame(rows)
    if rank.empty:
        return rank

    rank["us_rs63_pct"] = rank["us_ret63"].rank(pct=True)
    rank["us_sortino_pct"] = rank["us_sortino"].rank(pct=True)
    rank["us_score"] = (
        0.60 * rank["us_rs63_pct"] +
        0.20 * rank["us_sortino_pct"] +
        0.12 * rank["us_above11"].astype(int) +
        0.08 * rank["us_above21"].astype(int)
    )

    return rank.sort_values("us_score", ascending=False).reset_index(drop=True)


In [ ]:

def build_leaders_asof(
    kr_close_hist,
    kr_meta_hist,
    us_close_hist,
    asof_date,
    leader_count=LEADER_SECTOR_COUNT
):
    kr_rank = build_kr_etf_rank_asof(
        kr_close_hist, kr_meta_hist, asof_date
    )
    us_rank = build_us_sector_rank_asof(
        us_close_hist, asof_date
    )

    if kr_rank.empty:
        return pd.DataFrame(), kr_rank, us_rank

    reps = (
        kr_rank.sort_values("kr_score", ascending=False)
        .groupby("sector", as_index=False)
        .first()
        [["sector","ticker","name","kr_score","ret63","above_ma11","above_ma21"]]
        .rename(columns={
            "ticker":"rep_ticker",
            "name":"rep_name",
            "ret63":"kr_ret63"
        })
    )

    avgs = (
        kr_rank.groupby("sector", as_index=False)
        .agg(
            kr_sector_avg_score=("kr_score","mean"),
            kr_etf_count=("ticker","count")
        )
    )

    out = reps.merge(avgs, on="sector", how="left")

    if not us_rank.empty:
        out = out.merge(
            us_rank[["sector","us_score","us_ret63","us_above11","us_above21"]],
            on="sector",
            how="left"
        )

    if "us_score" not in out.columns:
        out["us_score"] = 0.0
    if "us_above11" not in out.columns:
        out["us_above11"] = False

    out["us_score"] = out["us_score"].fillna(0)
    out["us_above11"] = out["us_above11"].fillna(False)

    out["cross_market_confirm"] = (
        out["us_above11"] &
        (out["us_score"] >= 0.60)
    )

    out["final_sector_score"] = (
        0.70 * out["kr_score"] +
        0.15 * out["kr_sector_avg_score"] +
        0.15 * out["cross_market_confirm"].astype(int)
    )

    out["eligible"] = out["above_ma11"]

    leaders = (
        out[out["eligible"]]
        .sort_values("final_sector_score", ascending=False)
        .head(leader_count)
        .reset_index(drop=True)
    )

    leaders["asof_date"] = pd.Timestamp(asof_date)
    return leaders, kr_rank, us_rank


In [ ]:

def build_leader_history(
    kr_close_hist,
    kr_meta_hist,
    us_close_hist,
    start=None,
    end=None,
    freq="W-FRI",
):
    idx = kr_close_hist.index

    if start is not None:
        idx = idx[idx >= pd.Timestamp(start)]
    if end is not None:
        idx = idx[idx <= pd.Timestamp(end)]

    if len(idx) == 0:
        return pd.DataFrame()

    weekly_dates = (
        pd.Series(index=idx, dtype=float)
        .resample(freq)
        .last()
        .index
    )

    rows = []

    for d in weekly_dates:
        hist = idx[idx <= d]
        if len(hist) == 0:
            continue

        signal_date = hist[-1]

        leaders, _, _ = build_leaders_asof(
            kr_close_hist,
            kr_meta_hist,
            us_close_hist,
            signal_date
        )

        for rank_no, (_, r) in enumerate(leaders.iterrows(), 1):
            rows.append({
                "signal_date": signal_date,
                "rank": rank_no,
                "sector": r["sector"],
                "rep_ticker": r["rep_ticker"],
                "rep_name": r["rep_name"],
                "final_sector_score": r["final_sector_score"],
                "cross_market_confirm": r["cross_market_confirm"],
            })

    return pd.DataFrame(rows)


def leader_turnover_report(leader_history):
    if leader_history.empty:
        return pd.DataFrame()

    grouped = (
        leader_history.groupby("signal_date")["sector"]
        .apply(list)
        .sort_index()
    )

    rows = []
    prev = None

    for d, sectors in grouped.items():
        cur = set(sectors)

        if prev is None:
            turnover = np.nan
            added, removed = [], []
        else:
            added = sorted(cur - prev)
            removed = sorted(prev - cur)
            turnover = len(cur.symmetric_difference(prev)) / max(len(cur | prev), 1)

        rows.append({
            "signal_date": d,
            "sectors": ",".join(sectors),
            "added": ",".join(added),
            "removed": ",".join(removed),
            "turnover": turnover,
        })

        prev = cur

    return pd.DataFrame(rows)



# 14. 실행

## 실전
```python
daily = run_daily()
```

## 연구
```python
research = run_research()
```


In [ ]:
# daily = run_daily()
# research = run_research()

CONFIG["current_cash"] = 100_000_000
CONFIG["max_daily_turnover"] = 0.20
CONFIG["max_new_buys"] = 5

daily = run_daily()

=== HAA Leader System v2.0 / DAILY ===
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (char 0)
Error occurred in get_market_ticker_and_name: Expecting value: line 1 column 1 (c

KeyError: '시장'


# 15. v2.0 운영 원칙

### 매일
- `CONFIG["current_cash"]` 갱신
- `current_holdings.csv` 확인
- `daily = run_daily()`

### 주간
- 주도섹터 교체 확인
- 승인 주문표 검토
- ETF 구성종목 스냅샷 축적

### 월간
- NAV / MDD / 회전율 확인
- Research Mode로 Walk-forward 검증 업데이트
- Snapshot Coverage 확인

---

# 다음 v2.1 후보

v2.0 이후에는 대규모 구조변경보다 작은 개선이 적절합니다.

- 연구용 데이터 캐시 강화
- 주문 CSV를 증권사 업로드 형식에 맞춤
- 운영 리포트 HTML/PDF
- 알림 연동
- 보유종목 체결내역 자동 반영
